In [ ]:
# Please run this cell, to install the dependencies
# ! pip install pypdf openai pinecone


# Speak with PDF Assignment

**Assignment type:** RAG mini-app with three required functions  
**File to submit:** `pdf_ai.ipynb` (the notebook produced from this spec)  
**Language:** Python 3.x (Codio / Jupyter)  
**Time estimate:** 2–3 hours

---

## Objective

Build a **“Speak with PDF” RAG pipeline** that runs inside a Jupyter notebook and is compatible with **NBGrader**. You will implement **three functions**:

1) `parse_pdf_to_chunks(pdf_path, chunk_size=800, overlap=120) -> List[str]`  
2) `embed_and_index(chunks, use_pinecone=True, model="text-embedding-3-small") -> dict`  
3) `answer_query(question, index_obj, top_k=3, gpt_model="gpt-4o-mini") -> str`

**Grading will use OpenAI keys.** Learners must provide valid `OPENAI_API_KEY` (and Pinecone credentials if using Pinecone).

---

## What you will build

A tiny RAG system that:
- loads a PDF,
- extracts and **chunks** text,
- builds a **vector index** from OpenAI embeddings (Pinecone recommended; in‑memory/FAISS acceptable),
- answers a natural-language query by retrieving top‑k relevant chunks and generating an answer with an OpenAI chat model.

This mirrors production flow (PDF → chunks → embeddings → index → retrieval → answer).


---

## Tools & Libraries

- `openai` (for embeddings + chat completions)
- `pypdf` **or** `pdfplumber` (choose one) for text extraction
- Vector store: **Pinecone** (recommended) via `pinecone-client`; you may alternatively use **FAISS** or a simple in‑memory index for development
- `numpy`, `re`, `math`, `hashlib` (as needed)

> You may import additional libraries if helpful. Use your judgment.

**Environment variables expected (for grading):**
- `OPENAI_API_KEY`
- If using Pinecone: `PINECONE_API_KEY`, `PINECONE_ENV` (or project config), and an index name you create in notebook (e.g., `pdf-mini-app`).



---

## Function Requirements
You'll have to implment the following functionality down below

### 1) `parse_pdf_to_chunks(pdf_path, chunk_size=800, overlap=120) -> list[str]`

**Inputs**
- `pdf_path: str` — path to a readable PDF file (e.g., `data/sample.pdf`)
- `chunk_size: int` (default 800) — approximate number of characters per chunk
- `overlap: int` (default 120) — number of overlapping characters between adjacent chunks

**Output**
- `List[str]` — ordered, non-empty chunks of plain text. No chunk should exceed `chunk_size + 20` characters (allowing minor boundary variance).

**Behavioral constraints**
- Strip excessive whitespace and normalize newlines.
- Deterministic chunk boundaries given the same inputs.
- Apply overlap between chunks (except the first).

**Recommended approach**
1. Extract all PDF text into a single string using `pypdf` **or** `pdfplumber`.
2. Normalize whitespace.
3. Greedy chunk by character count with the requested overlap.



---

### 2) `embed_and_index(chunks, use_pinecone=True, model="text-embedding-3-small") -> dict`

**Purpose**  
Create embeddings with **OpenAI** and store them in a vector index (Pinecone preferred).

**Inputs**
- `chunks: List[str]` — the output of `parse_pdf_to_chunks`
- `use_pinecone: bool` — if `True`, upsert to Pinecone; if `False`, store locally (FAISS or simple in‑memory array search)
- `model: str` — OpenAI embedding model name (e.g., `"text-embedding-3-small"`)

**Output**
- `dict` with fields (minimum):
  - `"chunks"`: `List[str]`
  - `"index_backend"`: `"pinecone"` or `"local"`
  - `"index_ref"`: backend-specific handle (e.g., Pinecone index name / namespace, or local matrix + ids)
  - `"embed_model"`: embedding model used
  - `"dim"`: embedding dimensionality

**Notes**
- When `use_pinecone=True`: create (if needed) and upsert vectors with stable IDs (e.g., `doc-0`, `doc-1`, …). Keep a mapping from ID → original chunk text in the dict.
- When `use_pinecone=False`: store embeddings in memory (e.g., NumPy matrix) along with their chunk IDs for cosine‑similarity search.



---

### 3) `answer_query(question, index_obj, top_k=3, gpt_model="gpt-4o-mini") -> str`

**Purpose**  
Retrieve relevant chunks and generate an answer using an OpenAI chat model.

**Inputs**
- `question: str`
- `index_obj: dict` — the output from `embed_and_index`
- `top_k: int` — number of chunks to retrieve
- `gpt_model: str` — OpenAI chat model (e.g., `"gpt-4o-mini"`)

**Output**
- `str` — the final answer.

**Required behavior**
- Embed the **question** using the **same embedding model** used to build the index.
- Retrieve **top_k** chunks by cosine similarity (Pinecone query or local similarity search).
- Construct a concise prompt:
  - System: “You are a helpful assistant. Use ONLY the provided context to answer. If not answerable from context, respond: ‘I don’t know based on the document.’”
  - User: include the `question` and a `Context:` section concatenating the retrieved chunks (with simple separators).
- Call the OpenAI chat completion endpoint and **return the assistant’s message string**.



In [1]:
# This cell may be run by the grader
import os, re, math, hashlib
import numpy as np

In [2]:
import os
OPENROUTER_API_KEY = ''      # required
os.environ['PINECONE_API_KEY'] = ''      # optional (Gemini)

In [5]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

assert os.environ["OPENAI_API_KEY"].strip(), "Please set OPENAI_API_KEY"

USE_PINECONE = bool(os.getenv("PINECONE_API_KEY"))

Enter your OpenAI API key: ··········


<center><h1>--------------- Student Code ---------------</h1></center>

### B. Implement `parse_pdf_to_chunks`  **(student solution cell)**

### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [6]:
def parse_pdf_to_chunks(pdf_path, chunk_size=800, overlap=120):
    import re
    from pypdf import PdfReader

    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")

    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be >= 0 and < chunk_size")

    reader = PdfReader(pdf_path)

    text = ""
    for page in reader.pages:
        page_text = page.extract_text() or ""
        text += " " + page_text

    text = re.sub(r"\s+", " ", text).strip()

    if not text:
        return []

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(text), step):
        chunk = text[start:start + chunk_size].strip()

        if chunk:
            chunks.append(chunk)

    return chunks

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `parse_pdf_to_chunks`.
 -  If the function is not found, you answer will be rejected by the system

### C. Implement `embed_and_index`  **(student solution cell)**

### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [9]:
def embed_and_index(chunks, use_pinecone=True, model="text-embedding-3-small"):
    import os
    import numpy as np
    from openai import OpenAI

    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("Please set OPENAI_API_KEY first")

    client = OpenAI(api_key=api_key)

    chunks = [c.strip() for c in chunks if isinstance(c, str) and c.strip()]

    if not chunks:
        raise ValueError("chunks cannot be empty")

    embeddings = []

    # Create embeddings in batches
    for start in range(0, len(chunks), 100):
        batch = chunks[start:start + 100]

        response = client.embeddings.create(
            model=model,
            input=batch
        )

        ordered_data = sorted(response.data, key=lambda item: item.index)

        for item in ordered_data:
            embeddings.append(item.embedding)

    embeddings = np.array(embeddings, dtype=np.float32)

    ids = [f"doc-{i}" for i in range(len(chunks))]
    chunk_by_id = {
        ids[i]: chunks[i] for i in range(len(chunks))
    }

    if not use_pinecone:
        return {
            "chunks": chunks,
            "index_backend": "local",
            "index_ref": {
                "embeddings": embeddings,
                "ids": ids,
                "chunk_by_id": chunk_by_id
            },
            "embed_model": model,
            "dim": embeddings.shape[1],
            "client": client
        }

    # Pinecone is optional; configure it only when requested
    from pinecone import Pinecone, ServerlessSpec

    pinecone_key = os.getenv("PINECONE_API_KEY")
    if not pinecone_key:
        raise ValueError(
            "PINECONE_API_KEY is missing. Use use_pinecone=False for local indexing."
        )

    index_name = os.getenv("PINECONE_INDEX_NAME", "pdf-mini-app")
    pc = Pinecone(api_key=pinecone_key)

    existing = pc.list_indexes().names()

    if index_name not in existing:
        pc.create_index(
            name=index_name,
            dimension=embeddings.shape[1],
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region=os.getenv("PINECONE_REGION", "us-east-1")
            )
        )

    index = pc.Index(index_name)

    vectors = []
    for i in range(len(chunks)):
        vectors.append({
            "id": ids[i],
            "values": embeddings[i].tolist(),
            "metadata": {
                "chunk": chunks[i],
                "order": i
            }
        })

    for start in range(0, len(vectors), 100):
        index.upsert(vectors=vectors[start:start + 100])

    return {
        "chunks": chunks,
        "index_backend": "pinecone",
        "index_ref": index_name,
        "embed_model": model,
        "dim": embeddings.shape[1],
        "ids": ids,
        "chunk_by_id": chunk_by_id,
        "client": client
    }


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `embed_and_index`.
 -  If the function is not found, you answer will be rejected by the system

### D. Implement `answer_query`  **(student solution cell)**

### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [10]:
def answer_query(question, index_obj, top_k=3, gpt_model="gpt-4o-mini"):
    import numpy as np

    if not question or not question.strip():
        raise ValueError("Question cannot be empty")

    if top_k <= 0:
        raise ValueError("top_k must be positive")

    client = index_obj.get("client")
    if client is None:
        from openai import OpenAI
        client = OpenAI()

    # Embed the question using the same model as the document chunks
    model = index_obj["embed_model"]

    response = client.embeddings.create(
        model=model,
        input=[question.strip()]
    )

    query_vector = np.array(
        response.data[0].embedding,
        dtype=np.float32
    )

    retrieved_chunks = []

    if index_obj["index_backend"] == "local":
        ref = index_obj["index_ref"]
        matrix = np.asarray(ref["embeddings"], dtype=np.float32)

        if matrix.size == 0:
            return "I don’t know based on the document."

        # Calculate cosine similarity
        query_norm = np.linalg.norm(query_vector)
        matrix_norms = np.linalg.norm(matrix, axis=1)
        denominators = matrix_norms * query_norm

        similarities = np.zeros(len(matrix), dtype=np.float32)
        valid = denominators > 0
        similarities[valid] = (
            matrix[valid] @ query_vector
        ) / denominators[valid]

        # Stable sorting preserves earlier chunk order in ties
        ranked_indices = np.argsort(
            -similarities, kind="stable"
        )[:min(top_k, len(matrix))]

        ids = ref["ids"]
        chunk_by_id = ref["chunk_by_id"]

        retrieved_chunks = [
            chunk_by_id[ids[i]]
            for i in ranked_indices
            if ids[i] in chunk_by_id
        ]

    elif index_obj["index_backend"] == "pinecone":
        import os
        from pinecone import Pinecone

        api_key = os.getenv("PINECONE_API_KEY")
        if not api_key:
            raise ValueError("PINECONE_API_KEY is required")

        pc = Pinecone(api_key=api_key)
        index = pc.Index(index_obj["index_ref"])

        result = index.query(
            vector=query_vector.tolist(),
            top_k=top_k,
            include_metadata=True
        )

        matches = sorted(
            result.matches,
            key=lambda m: (
                -float(m.score or 0),
                int((m.metadata or {}).get("order", 10**12)),
                str(m.id)
            )
        )

        retrieved_chunks = [
            m.metadata["chunk"]
            for m in matches
            if m.metadata and m.metadata.get("chunk")
        ]

    else:
        raise ValueError("Unknown index backend")

    if not retrieved_chunks:
        return "I don’t know based on the document."

    context = "\n\n---\n\n".join(retrieved_chunks)

    system_prompt = (
        "You are a helpful assistant. Use ONLY the provided context "
        "to answer. If not answerable from context, respond: "
        "'I don’t know based on the document.'"
    )

    user_prompt = (
        f"Question: {question.strip()}\n\n"
        f"Context:\n{context}"
    )

    chat_response = client.chat.completions.create(
        model=gpt_model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0
    )

    answer = chat_response.choices[0].message.content

    return answer.strip() if answer else (
        "I don’t know based on the document."
    )

In [12]:
%pip install pypdf openai numpy

In [14]:
from google.colab import files

uploaded = files.upload()

Saving DNN_ST1_NN_MCQ_Paper.pdf to DNN_ST1_NN_MCQ_Paper.pdf


In [15]:
pdf_path = next(iter(uploaded.keys()))

print("PDF selected:", pdf_path)

PDF selected: DNN_ST1_NN_MCQ_Paper.pdf


In [17]:
chunks = parse_pdf_to_chunks(pdf_path)

print("Number of chunks:", len(chunks))
print(chunks[:2])

Number of chunks: 19
['Deep Neural Networks (24CAI0306) | Page 1 DEEP NEURAL NETWORKS (24CAI0306) Sample Paper: Numerical MCQs on Neural Networks (Sessional Test 1 Level) B.E. CSE (AI & ML), Semester 5 | Questions: 50 | Marks: 1 each Instructions: Each question has exactly one correct option. Use a calculator where required; round as indicated by the options. Answers with brief working are given at the end. Q1. An input matrix X of shape (32 × 784) is multiplied by a weight matrix W of shape (784 × 128). How many elements are there in the product XW? (a) 25,088 (b) 100,352 (c) 4,096 (d) 896 Q2. If y = (2x + 1)², then dy/dx at x = 1 is: (a) 12 (b) 6 (c) 18 (d) 9 Q3. Gradient descent is applied to L(θ) = θ² with θ = 5 and learning rate η = 0.1. The value of θ after one update is: (a) 4.5 (b) 4 (c) 5.1 (d) 0 Q4. Fo', 'o L(θ) = θ² with θ = 5 and learning rate η = 0.1. The value of θ after one update is: (a) 4.5 (b) 4 (c) 5.1 (d) 0 Q4. For L(θ) = (θ − 3)², start at θ0 = 0 with η = 0.25. The

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `answer_query`.
 -  If the function is not found, you answer will be rejected by the system


### E. (Optional) Demo cell (ungraded)
Provide a small demo that:
- Loads `data/sample.pdf`
- Runs all three functions
- Prints an example answer



---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  If the function is not found, you answer will be rejected by the system

---

## Hidden Tests (run by grader)

Hidden tests will verify that:

- `parse_pdf_to_chunks` applies overlap correctly, is deterministic, and honors size bounds.
- `embed_and_index` calls OpenAI embeddings and builds a valid index (Pinecone or local), with stable IDs and correct dimensionality/shape.
- Retrieval uses cosine similarity and returns the expected top‑k order on crafted inputs.
- `answer_query` embeds the question with the same model used for chunks, constructs a context‑bounded prompt, calls OpenAI chat completions, and returns a clean string.
- No hard‑coded secrets; keys must be read from environment.
- Notebook runs top‑to‑bottom reproducibly when keys are present.



---

## Rubric (100 pts)

- **PDF parsing & chunking (30 pts)**
  - Correct extraction and normalization (10)
  - Deterministic overlapping chunks within size bounds (15)
  - Edge cases (empty pages, weird whitespace) (5)

- **Embeddings & indexing (35 pts)**
  - Correct OpenAI embedding usage and shapes (15)
  - Proper index structure + metadata (10)
  - Stable IDs and retrievability (10)

- **Query flow & answer (25 pts)**
  - Correct cosine similarity retrieval and stable tie‑break (10)
  - Prompt enforces context‑only answering; sensible output (15)

- **Code quality & constraints (10 pts)**
  - Clean, readable, documented code (5)
  - Secure key handling; graceful error handling (5)



---

## (Optional) Pinecone/OpenAI Notes

- For Pinecone: create an index (e.g., 384 or 1536 dim depending on embedding model) and upsert vectors with metadata (`{"chunk": "...", "order": i}`).
- For local dev without Pinecone: store embeddings in a NumPy matrix and do cosine similarity. Grading is OK with either backend as long as OpenAI embeddings + chat are used.
- Keep your mapping from vector IDs to original chunk text; you’ll need it for context construction.



---

## Submission

Submit your completed notebook. Ensure it runs top‑to‑bottom without modification in a clean environment **with valid API keys set**. The grader will run hidden tests using OpenAI for embeddings and chat.


---

<h2><center> Completing your test </center></h2>

- Once you have completed your test and answered all the questions:
  
- You can open the guide using the button located on the far right of your screen.

- The button will look like this image:  
  ![](https://github.com/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%203/additional_files/End_guide_button.png?raw=1)

- Clicking on this button will allow you to mark the test as complete using the button shown below:  
  ![](https://github.com/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%203/additional_files/completion_button.png?raw=1)

- Please click on **Mark as Completed** to end and submit your test.

- Thank you!
